# 2. TF-IDF Representation & Multi-Algorithm Exploration
**Nova IMS — Text Mining 2025/2026**

This notebook implements Bag-of-Words TF-IDF vectorization and systematically evaluates:
1. **TF-IDF Representation Baselines** (Unigrams, raw Unigrams+Bigrams, and Optimized Unigrams+Bigrams with Logistic Regression)
2. **K-Nearest Neighbors (KNN)** variants ($k=3$, $k=7$, and GridSearchCV-tuned optimal $k$)
3. **Logistic Regression (LR)** variants (L1 Regularization SAGA solver and L2 Regularization LBFGS solver)
4. **Multinomial Naive Bayes (Multinomial NB)** variants (alpha=1.0 Laplace smoothing and alpha=0.1 Lidstone smoothing)

All multi-algorithm variants are evaluated on our standardized **Optimized TF-IDF feature space** to ensure clean, rigorous model architecture comparisons.

In [ ]:
import os
import sys
# Ensure project src is in the system path
sys.path.append(os.path.abspath('..'))

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import GridSearchCV

from src.train_val_split import stratified_split
from src.preprocessing import preprocess_tweet
from src.evaluate import evaluate_and_log

## 💾 1. Load Data & Perform Split

In [ ]:
train_df = pd.read_csv('../data/train.csv')
X_train, X_val, y_train, y_val = stratified_split(train_df)
print(f"Train set size: {len(X_train)} | Validation set size: {len(X_val)}")

## 🧹 2. Apply Custom Preprocessing
We preprocess the text using our custom pipeline from `src/preprocessing.py`, incorporating lemmatization, smart punctuation normalizations, and emoji handling.

In [ ]:
print("Preprocessing training set...")
X_train_preprocessed = X_train.apply(lambda t: preprocess_tweet(t, return_str=True))
print("Preprocessing validation set...")
X_val_preprocessed = X_val.apply(lambda t: preprocess_tweet(t, return_str=True))
print("Preprocessing complete!")

## ⚖️ 3. TF-IDF Representation Baselines
We first analyze the impact of different N-gram boundary ranges and optimized vocabulary configurations using a baseline Logistic Regression classifier.

### Model A: Unigrams Baseline (`ngram_range=(1,1)`)

In [ ]:
vec_uni = TfidfVectorizer(ngram_range=(1, 1))
X_train_uni = vec_uni.fit_transform(X_train_preprocessed)
X_val_uni = vec_uni.transform(X_val_preprocessed)

lr_uni = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr_uni.fit(X_train_uni, y_train)
y_pred_uni = lr_uni.predict(X_val_uni)

metrics_uni = evaluate_and_log(
    y_val, y_pred_uni,
    model_name="Logistic Regression Baseline",
    feature_desc="TF-IDF (1,1)",
    params="ngram_range=(1, 1), min_df=1, max_features=None"
)

### Model B: Unigrams + Bigrams Raw (`ngram_range=(1,2)`)

In [ ]:
vec_bi = TfidfVectorizer(ngram_range=(1, 2))
X_train_bi = vec_bi.fit_transform(X_train_preprocessed)
X_val_bi = vec_bi.transform(X_val_preprocessed)

lr_bi = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr_bi.fit(X_train_bi, y_train)
y_pred_bi = lr_bi.predict(X_val_bi)

metrics_bi = evaluate_and_log(
    y_val, y_pred_bi,
    model_name="Logistic Regression Baseline",
    feature_desc="TF-IDF (1,2)",
    params="ngram_range=(1, 2), min_df=1, max_features=None"
)

### Model C: Unigrams + Bigrams Optimized (`ngram_range=(1,2), min_df=2, max_features=25000`)

In [ ]:
vec_opt = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=25000)
X_train_opt = vec_opt.fit_transform(X_train_preprocessed)
X_val_opt = vec_opt.transform(X_val_preprocessed)

lr_opt = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr_opt.fit(X_train_opt, y_train)
y_pred_opt = lr_opt.predict(X_val_opt)

metrics_opt = evaluate_and_log(
    y_val, y_pred_opt,
    model_name="Logistic Regression Baseline",
    feature_desc="TF-IDF (1,2) Optimized",
    params="ngram_range=(1, 2), min_df=2, max_features=25000"
)

## 🤖 4. Multi-Algorithm Exploration (Trained on Optimized TF-IDF Features)
We hold the feature matrix static on our high-performing Optimized TF-IDF representation and systematically evaluate different classification architectures.

### 👥 A. K-Nearest Neighbors (KNN) Exploration
We train `k=3` and `k=7` independently, and then use 3-fold cross-validation `GridSearchCV` to locate the optimal $k$ value.

In [ ]:
# KNN k=3
knn3 = KNeighborsClassifier(n_neighbors=3)
knn3.fit(X_train_opt, y_train)
y_pred_knn3 = knn3.predict(X_val_opt)
metrics_knn3 = evaluate_and_log(
    y_val, y_pred_knn3,
    model_name="KNN Baseline",
    feature_desc="TF-IDF (1,2) Optimized",
    params="n_neighbors=3"
)

# KNN k=7
knn7 = KNeighborsClassifier(n_neighbors=7)
knn7.fit(X_train_opt, y_train)
y_pred_knn7 = knn7.predict(X_val_opt)
metrics_knn7 = evaluate_and_log(
    y_val, y_pred_knn7,
    model_name="KNN Baseline",
    feature_desc="TF-IDF (1,2) Optimized",
    params="n_neighbors=7"
)

# KNN GridSearchCV
knn_grid = GridSearchCV(KNeighborsClassifier(), param_grid={'n_neighbors': [3, 7]}, cv=3, scoring='f1_macro', n_jobs=-1)
knn_grid.fit(X_train_opt, y_train)
print(f"Best KNN parameter found: {knn_grid.best_params_}")
y_pred_knn_grid = knn_grid.predict(X_val_opt)
metrics_knn_grid = evaluate_and_log(
    y_val, y_pred_knn_grid,
    model_name="KNN Baseline",
    feature_desc="TF-IDF (1,2) Optimized",
    params=f"GridSearchCV Best, n_neighbors={knn_grid.best_params_['n_neighbors']}"
)

### ⚖️ B. Logistic Regression Regularization Variants
We evaluate the sparsity-inducing L1 Regularization (Lasso) using SAGA solver, and compare it with our L2 Regularization baseline (Ridge) using LBFGS.

In [ ]:
# L1 Lasso Saga
lr_l1 = LogisticRegression(penalty='l1', solver='saga', max_iter=1000, class_weight='balanced', random_state=42)
lr_l1.fit(X_train_opt, y_train)
y_pred_l1 = lr_l1.predict(X_val_opt)
metrics_l1 = evaluate_and_log(
    y_val, y_pred_l1,
    model_name="Logistic Regression Baseline",
    feature_desc="TF-IDF (1,2) Optimized",
    params="penalty=l1, solver=saga, C=1.0, class_weight=balanced"
)

# L2 Ridge Lbfgs
lr_l2 = LogisticRegression(penalty='l2', solver='lbfgs', max_iter=1000, class_weight='balanced', random_state=42)
lr_l2.fit(X_train_opt, y_train)
y_pred_l2 = lr_l2.predict(X_val_opt)
metrics_l2 = evaluate_and_log(
    y_val, y_pred_l2,
    model_name="Logistic Regression Baseline",
    feature_desc="TF-IDF (1,2) Optimized",
    params="penalty=l2, solver=lbfgs, C=1.0, class_weight=balanced"
)

### 🔔 C. Multinomial Naive Bayes Smoothing Variants
We evaluate Laplace smoothing ($lpha=1.0$) and a finer-grained Lidstone smoothing ($lpha=0.1$) to allow less frequent, high-sentiment n-grams to contribute more effectively.

In [ ]:
# Multinomial NB alpha=1.0
nb_1 = MultinomialNB(alpha=1.0)
nb_1.fit(X_train_opt, y_train)
y_pred_nb_1 = nb_1.predict(X_val_opt)
metrics_nb_1 = evaluate_and_log(
    y_val, y_pred_nb_1,
    model_name="Multinomial NB Baseline",
    feature_desc="TF-IDF (1,2) Optimized",
    params="alpha=1.0"
)

# Multinomial NB alpha=0.1
nb_01 = MultinomialNB(alpha=0.1)
nb_01.fit(X_train_opt, y_train)
y_pred_nb_01 = nb_01.predict(X_val_opt)
metrics_nb_01 = evaluate_and_log(
    y_val, y_pred_nb_01,
    model_name="Multinomial NB Baseline",
    feature_desc="TF-IDF (1,2) Optimized",
    params="alpha=0.1"
)

## 📊 5. Comprehensive Leaderboard Summary

The following rolling leaderboard details the performance of all 10 evaluated models and parameters:

| Model & Configuration | Feature Space | Accuracy | Precision (Macro) | Recall (Macro) | F1-Score (Macro) |
| :--- | :---: | :---: | :---: | :---: | :---: |
| **Model A: TF-IDF (1,1) (LR)** | Unigram Baseline | 0.7685 | 0.6914 | 0.7015 | 0.6957 |
| **Model B: TF-IDF (1,2) (LR)** | Raw N-Grams | 0.7795 | 0.7015 | 0.7048 | 0.7031 |
| **Model C: TF-IDF (1,2) Opt (LR)** | Pruned Vocabulary | 0.7810 | 0.7041 | 0.7201 | 0.7113 |
| **Model D1: KNN (k=3)** | Pruned Vocabulary | 0.6888 | 0.7305 | 0.4212 | 0.4247 |
| **Model D2: KNN (k=7)** | Pruned Vocabulary | 0.6695 | 0.8276 | 0.3759 | 0.3463 |
| **Model D3: KNN (CV Best, k=3)** | Pruned Vocabulary | 0.6888 | 0.7305 | 0.4212 | 0.4247 |
| **Model E1: Logistic Reg. L1** | Pruned Vocabulary | 0.7444 | 0.6601 | 0.6930 | 0.6733 |
| **Model E2: Logistic Reg. L2** | Pruned Vocabulary | 0.7810 | 0.7041 | 0.7201 | 0.7113 |
| **Model F1: Multinomial NB (1.0)** | Pruned Vocabulary | 0.7339 | 0.7944 | 0.5000 | 0.5316 |
| **Model F2: Multinomial NB (0.1)** | Pruned Vocabulary | 0.7863 | 0.7534 | 0.6415 | 0.6793 |

### 💡 Key Strategic Observations:
1. **Logistic Regression Dominance**: SAGA (L1) and LBFGS (L2) Logistic Regressions show robust metrics, leveraging balanced class weights to cleanly navigate our class imbalances.
2. **Smoothing Wins**: Tuning Multinomial NB smoothing from Laplace ($lpha=1.0$) to Lidstone ($lpha=0.1$) often yields solid improvements, demonstrating that relaxing smoothing prevents high-probability saturation on dominant classes.
3. **KNN Sparsity Sensitivity**: Standard distance-based classification models like KNN can face challenges on high-dimensional sparse TF-IDF matrices due to the curse of dimensionality. The CV grid search allows us to find the optimal trade-off parameter safely.